In [ ]:
import pandas as pd
import fsspec
from google.oauth2 import service_account

import dask.dataframe as dd
import xarray as xr
from shapely import wkb
import geopandas as gpd
from xclim.indices.stats import standardized_index_fit_params
from xclim.indices import standardized_precipitation_index

import xesmf as xe
import numpy as np
# Path to your service account key file
service_account_json =  '../../coiled-data-key.json'
# Create credentials object
credentials = service_account.Credentials.from_service_account_file(
    service_account_json,
    scopes=["https://www.googleapis.com/auth/devstorage.read_only"],
  # Adjust scopes if needed
)

In [ ]:
# GCS file URL
gcs_file_url = 'gs://seas51/ea_admin0_2_custom_polygon_shapefile_v5.parquet'
ddf = dd.read_parquet(gcs_file_url, storage_options={'token': credentials}, engine='pyarrow')
fdf = ddf[ddf['gbid'].str.contains('kmj')]
df1 = fdf.compute()
df1['geometry'] = df1['geometry'].apply(wkb.loads)
gdf = gpd.GeoDataFrame(df1, geometry='geometry')

In [ ]:
bounds=gdf.bounds
lat_min = bounds['miny'].min()
lat_max = bounds['maxy'].max()
lon_min = bounds['minx'].min()
lon_max = bounds['maxx'].max()

In [ ]:
# GCS path to the Zarr file
gcs_zarr_path = "gs://seas51/chirps_v2_monthly_20241012.zarr"
fs = fsspec.filesystem("gs", token=credentials)
# Get the mapper
m = fs.get_mapper(gcs_zarr_path)
# Open the dataset using xarray
ds = xr.open_zarr(m, consolidated=False)
# Define the polygon boundaries
#lon1, lat1 = 32.0, 1.0
#lon2, lat2 = 36.0, 5.0
# Extract the data for the specified polygon
ds_subset = ds.sel(latitude=slice(lat_min, lat_max), longitude=slice(lon_min, lon_max))
# Compute the data locally
ds_subset_computed = ds_subset.compute()
ds_subset_computed

In [ ]:
dsc=ds_subset_computed
ds1=dsc.rename({'longitude':'lon','latitude':'lat'})
dr = ds1["precip"] 
ds_out = xr.Dataset(
      {"lat": (["lat"], np.arange(lat_min, lat_max, 0.25), {"units": "degrees_north"}),
      "lon": (["lon"], np.arange(lon_min, lon_max, 0.25), {"units": "degrees_east"}),})
regridder = xe.Regridder(ds1, ds_out, "bilinear")
regridder  # print basic regridder information.
dr_out = regridder(dr, keep_attrs=True)
ds2=dr_out.to_dataset()
ds2.to_netcdf(f'ea-km25-chirps-v2.0.monthly.nc')

In [ ]:
# Calculate the total size of the dataset in bytes
total_size = ds2.nbytes
print(f"Total data size: {total_size / 1e6:.2f} MB")

In [ ]:
ds2

In [ ]:
ch_ds=ds2
ch_ds['precip'].attrs['units'] = 'mm/month'
#ch_ds1 = ch_ds.chunk(-1)
aa=ch_ds.precip
spi_3 = standardized_precipitation_index(
     aa,
     freq="MS",
     window=3,
     dist="gamma",
     method="APP",
     cal_start='1991-01-01',
     cal_end='2018-01-01',
    fitkwargs={"floc": 0} 
)

In [ ]:
a_s3=spi_3.compute()

In [ ]:
a_s3

In [ ]:
ch_spi= a_s3.to_dataset(name='spi3')

total_size = ch_spi.nbytes
print(f"Total data size: {total_size / 1e6:.2f} MB")

In [ ]:
ch_spi['time']

In [ ]:
#ds2=dr_out.to_dataset()
ch_spi.to_netcdf(f'kmj-25km-chirps-v2.0.monthly.nc')

## seas51

In [ ]:
import xarray as xr
import fsspec
from google.oauth2 import service_account

# GCS path to the Zarr file
gcs_zarr_path = "gs://seas51/seas51_20241012_v3.zarr"

# Define path to your credentials JSON file
credentials_path = "../../coiled-data-key.json"

# Specify the correct GCS scope
scopes = ["https://www.googleapis.com/auth/devstorage.read_write"]

# Create credentials object with the required scope
credentials = service_account.Credentials.from_service_account_file(
    credentials_path, scopes=scopes
)

# Create an fsspec filesystem with GCS using the service account
fs = fsspec.filesystem("gs", token=credentials)

# Get the mapper
m = fs.get_mapper(gcs_zarr_path)

# Open the dataset using xarray
sds = xr.open_zarr(m, consolidated=False)

# Extract the data for the specified polygon
sds_subset = sds.sel(latitude=slice(lat_max, lat_min), longitude=slice(lon_min, lon_max))

# Compute the data locally
sds_subset_computed = sds_subset.compute()

In [ ]:
sds_subset_computed

In [ ]:
def apply_spi3(cont_db,lead_val):
    """
    Calculates the 3-month Standardized Precipitation Index (SPI) for a specified lead time.

    Parameters:
    - cont_db (xarray.Dataset): The input dataset containing total monthly precipitation data.
    - lead_val (int): The lead time value for which the SPI is calculated.

    Returns:
    - cont_spi (list): A list of xarray.DataArrays containing the SPI values for each ensemble member.
    """
    lt1_db = cont_db.sel(forecastMonth=lead_val)
    lt1_db['tprate'].attrs['units'] = 'mm/month'
    cont_spi=[]
    for nsl in lt1_db.number.values:
        lt1_db2=lt1_db.sel(number=nsl)
        #lt1_db3 = lt1_db2.chunk({'time': 4, 'latitude': 2, 'longitude': 2})
        #lt1_db3 = lt1_db2.chunk(-1)
        aa=lt1_db2.tprate
        spi_3 = standardized_precipitation_index(
             aa,
             freq="MS",
             window=3,
             dist="gamma",
             method="APP",
             cal_start='1991-01-01',
             cal_end='2018-01-01',
             fitkwargs={"floc": 0} 
        )  
        a_s3=spi_3.compute()
        cont_spi.append(a_s3)
        aa=[]
        lt1_db3 = []
        lt1_db2 = []
        print(nsl)
    return cont_spi

In [ ]:
lead_val=1
cont1=apply_spi3(sds_subset_computed,lead_val)
t1d = xr.concat(cont1, dim='member')
t1da= t1d.to_dataset(name='spi3')

lead_val=2
cont2=apply_spi3(sds_subset_computed,lead_val)
t2d = xr.concat(cont2, dim='member')
t2da= t2d.to_dataset(name='spi3')
print('lt2')

lead_val=3
cont3=apply_spi3(sds_subset_computed,lead_val)
t3d = xr.concat(cont3, dim='member')
t3da= t3d.to_dataset(name='spi3')
print('lt3')

lead_val=4
cont4=apply_spi3(sds_subset_computed,lead_val)
t4d = xr.concat(cont4, dim='member')
t4da= t4d.to_dataset(name='spi3')
print('lt4')

lead_val=5
cont5=apply_spi3(sds_subset_computed,lead_val)
t5d = xr.concat(cont5, dim='member')
t5da= t5d.to_dataset(name='spi3')
print('lt5')

lead_val=6
cont6=apply_spi3(sds_subset_computed,lead_val)
t6d = xr.concat(cont6, dim='member')
t6da= t6d.to_dataset(name='spi3')
print('lt6')

In [ ]:
ds_ea = xr.concat([t1d,t2d,t3d,t4d,t5d,t6d], dim='lead')
ds_ea= ds_ea.to_dataset(name='spi3')
ds_ea.spi3.nbytes / (1024*1024)
ds_ea.to_netcdf(f'kmj_nr_seas51_spi3_xclim_20241105.nc')

In [ ]:
#kn_obs=xr.open_dataset(f'kmj-nr-chirps-v2.0.monthly.nc')
#kn_obs

In [ ]:
#ds_p_m1=ds_ea.sel(lead=1)
#ds_p_m1

In [ ]:
#fcst=xr.open_dataset(f'{data_path}ea_seas51_spi3_xclim_20240306.nc')
kn_fc=ds_ea


fc_llon = kn_fc.longitude.min().values
fc_ulon = kn_fc.longitude.max().values
fc_llat = kn_fc.latitude.min().values
fc_ulat = kn_fc.latitude.max().values

#obs=xr.open_dataset(f'{data_path}ea_chrips_spi3_xclim_20240306.nc')
kn_obs=xr.open_dataset('kmj-25km-chirps-v2.0.monthly.nc')
#kn_obs['lon']

cont_d=[]

for fm in [0,1,2,3,4,5]:
    ds_p_m1=kn_fc.sel(lead=fm)
    ds_out = xr.Dataset(
          {"lat": (["lat"], kn_obs['lat'].values, {"units": "degrees_north"}),
          "lon": (["lon"], kn_obs['lon'].values, {"units": "degrees_east"}),})
    gd2=ds_p_m1.rename({'longitude':'lon','latitude':'lat'})
    agd = gd2["spi3"]
    regridder = xe.Regridder(gd2, ds_out, "bilinear")
    dr_out = regridder(agd, keep_attrs=True)
    ds2=dr_out.to_dataset()
    cont_d.append(ds2)
    #monthname=mnl.lower().split('.')[0]
    #ds2.to_netcdf(f'{output_path_location}kmj_25km_lt_month_{fm}.nc')
    
kn_fct = xr.concat(cont_d, dim='lead')
kn_fct=kn_fct.rename({'time':'init','forecastMonth':'lead'})
#ld1 = ld.rename({'time':'init','forecastMonth':'lead'}).set_index(init='time', lead='forecastMonth')
#ld1 = ld.swap_dims({'time': 'init', 'forecastMonth': 'lead'})
kn_fct['lead'].attrs['units'] = 'months'


#output
kn_fct.to_netcdf(f'kn_fct_spi3.nc')
#kn_obs.to_netcdf(f'{data_path}kn_obs_spi3.nc')

In [ ]:
kn_obs

In [ ]:
kn_fct